In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-4-1-gene-ratio-different-tissue

Summarize tissue gene ratios and word clouds.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


Mouse tissue gene ratios.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import re
import mygene

# ==========================================

# ==========================================
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
mpl.rcParams['axes.linewidth'] = 1.2


def match_keywords(text, keywords):
    for k in keywords:

        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

# ==========================================

# ==========================================
def get_objective_classifications(gene_list):
    if not gene_list:
        print("警告：未读取到任何基因，请检查 txt 文件路径。")
        return {}

    print(f"正在深度联网查询 {len(gene_list)} 个基因的名称与 GO 功能库，请稍候...")
    mg = mygene.MyGeneInfo()
    

    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='mouse')

    gene_info_dict = {}
    for res in results:
        query_gene = res.get('query')
        if query_gene and query_gene not in gene_info_dict:
            gene_info_dict[query_gene] = res

    classification_dict = {}
    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()


            go_terms = ""
            if 'go' in row:
                go_data = row['go']
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in go_data:
                        items = go_data[sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items:
                                go_terms += str(item.get('term', '')).lower() + " "
            

            search_text = desc + " " + go_terms + " " + gene_type


            

            if 'rna' in gene_type or 'pseudo' in gene_type or re.match(r'^GM\d+$', symbol) or symbol.endswith('RIK') or re.match(r'^BC\d+$', symbol):
                category = 'ncRNA & Uncharacterized'
            

            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or \
                 match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
                

            elif match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
                

            elif match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
                

            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
                

            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
                

            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
                

            elif match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
                

            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
                

            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
                

            elif match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
                

            elif match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
                

            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================

# ==========================================
folder_path = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")
all_genes = set()
tissue_genes = {}

for file in os.listdir(folder_path):
    if file.endswith('.txt') and "Knee_Genes" in file:
        tissue_name = re.sub(r'_Knee_Genes_\d+\.txt', '', file).replace('_', ' ')
        with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
            genes = [line.strip().split()[-1].upper() for line in f.readlines() if line.strip()]
            tissue_genes[tissue_name] = genes
            all_genes.update(genes)

# ==========================================

# ==========================================
global_gene_dict = get_objective_classifications(list(all_genes))


categories = [
    'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
    'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
    'Transcription Regulation', 'Development & Differentiation', 
    'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
    'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
]

results = []
for tissue, genes in tissue_genes.items():
    if not genes: continue
    counts = {cat: 0 for cat in categories}
    for g in genes:
        cat = global_gene_dict.get(g, 'Other / Unclassified')
        counts[cat] += 1

    total = sum(counts.values())
    perc = {cat: (counts[cat]/total)*100 for cat in categories}
    perc['Tissue'] = tissue
    results.append(perc)

df = pd.DataFrame(results)
if df.empty:
    print("未生成任何数据，可能是当前目录下没有找到符合 '_Knee_Genes_' 命名的 txt 文件！")
else:
    df.set_index('Tissue', inplace=True)

    df = df.sort_values(by='Ribosome, RNA & Translation', ascending=False)
    

    df.to_csv(os.path.join(folder_path, "Gene_Classification_Data.csv"))
    pd.DataFrame.from_dict(global_gene_dict, orient='index', columns=['Category']).to_csv(os.path.join(folder_path, "Gene_to_Category_Map.csv"))

    # ==========================================

    # ==========================================
    fig, ax = plt.subplots(figsize=(16, 8))

    colors = [
        '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
        '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
    ]

    ax.yaxis.grid(True, color='#D3D3D3', linestyle='--', linewidth=0.7, zorder=0)

    df[categories].plot(kind='bar', stacked=True, ax=ax, color=colors, 
                        width=0.8, edgecolor='white', linewidth=0.7, zorder=3)

    ax.set_ylim(0, 100)
    ax.set_ylabel('Percentage of Genes (%)', fontsize=14, fontweight='bold', labelpad=10)
    ax.set_xlabel('', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    legend = ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=11, 
                       frameon=False, title='Gene Functional Class', title_fontsize=13, labelspacing=0.8)
    legend.get_title().set_fontweight('bold')

    plt.tight_layout()
    plt.savefig(os.path.join(folder_path, "Aging_Genes_Proportions_Final.pdf"))
    plt.savefig(os.path.join(folder_path, "Aging_Genes_Proportions_Final.png"), dpi=300)
    print("✅ 深度 GO 词元正则归类完毕！图表及分类字典映射表 (CSV) 已生成。")
    plt.show()

Human tissue gene ratios.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import re
import mygene

# ==========================================

# ==========================================
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
mpl.rcParams['axes.linewidth'] = 1.2


def match_keywords(text, keywords):
    for k in keywords:

        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

# ==========================================

# ==========================================
def get_human_gene_classifications(gene_list):
    if not gene_list:
        print("警告：未读取到任何基因，请检查 txt 文件路径。")
        return {}

    print(f"正在深度联网查询 {len(gene_list)} 个人类基因的名称与 GO 功能库，请稍候...")
    mg = mygene.MyGeneInfo()
    

    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='human')

    gene_info_dict = {}
    for res in results:
        query_gene = res.get('query')
        if query_gene and query_gene not in gene_info_dict:
            gene_info_dict[query_gene] = res

    classification_dict = {}
    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()


            go_terms = ""
            if 'go' in row:
                go_data = row['go']
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in go_data:
                        items = go_data[sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items:
                                go_terms += str(item.get('term', '')).lower() + " "
            

            search_text = desc + " " + go_terms + " " + gene_type


            

            if 'rna' in gene_type or 'pseudo' in gene_type or symbol.startswith(('MIR', 'LINC', 'LOC')):
                category = 'ncRNA & Uncharacterized'
            

            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or \
                 match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
                

            elif symbol.startswith(('COL', 'MMP', 'ACT', 'MYO', 'KRT')) or \
                 match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
                

            elif symbol.startswith(('HLA', 'CXCL', 'CCL', 'IL', 'CD', 'IG')) or \
                 match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
                

            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
                

            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
                

            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
                

            elif symbol.startswith(('ZNF', 'STAT', 'SMAD', 'HOX', 'FOX')) or \
                 match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
                

            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
                

            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
                

            elif 'R' in symbol[-2:] or match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
                

            elif symbol.startswith(('SLC', 'ATP', 'NDUF')) or \
                 match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
                

            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================

# ==========================================

folder_path = input_path("2-8.3-shanda/1-feature/1-human-guaidian-choose-gene/Gene_Lists")

all_genes = set()
tissue_genes = {}


if not os.path.exists(folder_path):
    print(f"❌ 找不到路径: {folder_path}，请修改代码中的 folder_path！")
else:
    for file in os.listdir(folder_path):
        if file.endswith('.txt'):

            name_clean = file

            name_clean = re.sub(r'^type_\d+_', '', name_clean, flags=re.IGNORECASE)

            name_clean = re.sub(r'_Knee_\d+_Genes\.txt$', '', name_clean, flags=re.IGNORECASE)

            name_clean = re.sub(r'_(Knee|Genes|Gene).*$', '', name_clean, flags=re.IGNORECASE)
            name_clean = name_clean.replace('.txt', '')


            tissue_name = name_clean.replace('_', ' ').title()
            # -------------------------------------------------------------------------

            file_path = os.path.join(folder_path, file)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:

                genes = [line.strip().split()[-1].upper() for line in f.readlines() if line.strip()]
                tissue_genes[tissue_name] = genes
                all_genes.update(genes)

    human_gene_dict = get_human_gene_classifications(list(all_genes))

    # ==========================================

    # ==========================================
    categories = [
        'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
        'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
        'Transcription Regulation', 'Development & Differentiation', 
        'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
        'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
    ]

    results = []
    for tissue, genes in tissue_genes.items():
        if not genes: continue
        counts = {cat: 0 for cat in categories}
        for g in genes:
            cat = human_gene_dict.get(g, 'Other / Unclassified')
            counts[cat] += 1

        total = sum(counts.values())
        perc = {cat: (counts[cat]/total)*100 for cat in categories}
        perc['Tissue'] = tissue
        results.append(perc)

    df = pd.DataFrame(results)
    
    if not df.empty:
        df.set_index('Tissue', inplace=True)

        df = df.sort_values(by='Ribosome, RNA & Translation', ascending=False)

        # ==========================================

        # ==========================================
        fig, ax = plt.subplots(figsize=(16, 8))


        colors = [
            '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
            '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
        ]
        ax.yaxis.grid(True, color='#D3D3D3', linestyle='--', linewidth=0.7, zorder=0)

        df[categories].plot(kind='bar', stacked=True, ax=ax,
                            color=colors, width=0.8,
                            edgecolor='white', linewidth=0.7, zorder=3)

        ax.set_ylim(0, 100)
        ax.set_ylabel('Percentage of Genes (%)', fontsize=14, fontweight='bold', labelpad=10)
        ax.set_xlabel('', fontsize=14)
        plt.xticks(rotation=45, ha='right', fontsize=12)
        plt.yticks(fontsize=12)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        legend = ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
                           fontsize=11, frameon=False, title='Gene Functional Class',
                           title_fontsize=13, labelspacing=0.8)
        legend.get_title().set_fontweight('bold')

        plt.tight_layout()
        

        output_pdf = os.path.join(folder_path, "1-figure-Human_Gene_Classification_Proportions.pdf")

        output_png = os.path.join(folder_path, "1-figure-Human_Gene_Classification_Proportions.png")


        plt.savefig(output_pdf, bbox_inches='tight') 
        plt.savefig(output_png, dpi=300, bbox_inches='tight')

        print(f"✅ 图表已成功生成！")
        print(f"📄 PDF 矢量图已保存至: {output_pdf}")
        print(f"🖼️ PNG 预览图已保存至: {output_png}")
        plt.show()
    else:
        print("没有可用的数据生成图表，请检查文本文件。")

Gene word-cloud plots.

In [ ]:
import os
import pandas as pd
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from PIL import Image


INPUT_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")
OUTPUT_DIR = input_path("2-8.3-shanda/1-feature/9-Summary_Results")




COLOR_MAP = 'plasma'
MAX_WORDS = 100
DPI = 300


os.makedirs(OUTPUT_DIR, exist_ok=True)

def create_oval_mask(width=1600, height=900):
    """Create an oval mask array."""
    x, y = np.ogrid[:height, :width]
    center_x, center_y = width / 2, height / 2


    mask = ((x - center_y) ** 2 / (height * 0.45) ** 2 +
            (y - center_x) ** 2 / (width * 0.45) ** 2) > 1
    return 255 * mask.astype(int)

if not os.path.exists(INPUT_DIR):
    print(f"错误: 找不到目录 {INPUT_DIR}")
else:
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')]
    all_genes = []

    for filename in files:
        file_path = os.path.join(INPUT_DIR, filename)
        with open(file_path, 'r') as f:
            all_genes.extend([line.strip().upper() for line in f if line.strip()])

    if not all_genes:
        print("未找到基因数据。")
    else:

        gene_freq = dict(Counter(all_genes))


        print(f"正在生成主刊级椭圆词云 (Color-safe: {COLOR_MAP})...")

        oval_mask = create_oval_mask(2000, 1200)

        wc = WordCloud(
            width=2000, height=1200,
            background_color='white',
            mask=oval_mask,
            colormap=COLOR_MAP,
            max_words=MAX_WORDS,
            min_font_size=10,
            max_font_size=250,
            random_state=42,
            prefer_horizontal=0.9,
            relative_scaling=0.5,
            contour_width=1,
            contour_color='#eeeeee'
        )

        wc.generate_from_frequencies(gene_freq)


        plt.figure(figsize=(20, 12))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")


        out_base = os.path.join(OUTPUT_DIR, "Aging_Gene_Cloud_Oval")

        plt.savefig(f"{out_base}.pdf", dpi=DPI, format='pdf', bbox_inches='tight', pad_inches=0.1)

        plt.savefig(f"{out_base}.png", dpi=DPI, format='png', bbox_inches='tight', pad_inches=0.1)

        plt.show()

        print("\n" + "="*40)
        print(f"✅ 处理完成！")
        print(f"1. 椭圆词云 (PDF/PNG) 已保存至: {OUTPUT_DIR}")
        print(f"2. 使用色图: {COLOR_MAP} (红绿色盲友好)")
        print("="*40)

Gene word-cloud plots.

In [ ]:
import os
import pandas as pd
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import platform



INPUT_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")
OUTPUT_DIR = output_path("2-8.3-shanda/1-feature/1-figure/2-ciyuntu")


system = platform.system()
if system == "Windows":
    FONT_PATH = sage_font_path()
elif system == "Darwin": # macOS
    FONT_PATH = sage_font_path()
else: # Linux / WSL

    wsl_font_path = sage_font_path()
    linux_native_path = sage_font_path()
    
    if os.path.exists(wsl_font_path):
        FONT_PATH = wsl_font_path
    elif os.path.exists(linux_native_path):
        FONT_PATH = linux_native_path
    else:

        print("⚠️ 未能在系统路径中找到 Arial 字体。正在尝试使用当前目录下的 arial.ttf...")
        FONT_PATH = "arial.ttf" 


COLOR_MAP = 'plasma'
MAX_WORDS = 100 
DPI = 300


os.makedirs(OUTPUT_DIR, exist_ok=True)

def create_oval_mask(width=2000, height=1200):
    """Create an oval mask array."""
    x, y = np.ogrid[:height, :width]
    center_x, center_y = width / 2, height / 2

    mask = ((x - center_y) ** 2 / (height * 0.45) ** 2 +
            (y - center_x) ** 2 / (width * 0.45) ** 2) > 1
    return 255 * mask.astype(int)

if not os.path.exists(INPUT_DIR):
    print(f"错误: 找不到目录 {INPUT_DIR}")
else:
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')]
    all_genes = []

    for filename in files:
        file_path = os.path.join(INPUT_DIR, filename)
        with open(file_path, 'r') as f:

            all_genes.extend([line.strip().capitalize() for line in f if line.strip()])

    if not all_genes:
        print("未找到基因数据。")
    else:
        gene_freq = dict(Counter(all_genes))

        print(f"正在生成主刊级椭圆词云 (Color-safe: {COLOR_MAP})...")

        oval_mask = create_oval_mask(2000, 1200) 

        try:
            wc = WordCloud(
                font_path=FONT_PATH,
                width=2000, height=1200,
                background_color='white',
                mask=oval_mask,
                colormap=COLOR_MAP,
                max_words=MAX_WORDS,
                min_font_size=12,
                max_font_size=250,      
                random_state=42,
                prefer_horizontal=1.0,  
                relative_scaling=0.5,   
                contour_width=0,        
            )

            wc.generate_from_frequencies(gene_freq)


            plt.figure(figsize=(7, 4.2))
            plt.imshow(wc, interpolation="bilinear")
            plt.axis("off")


            out_base = os.path.join(OUTPUT_DIR, "Aging_Gene_Cloud_Oval")
            

            plt.savefig(f"{out_base}.png", dpi=DPI, format='png', bbox_inches='tight', pad_inches=0.1)
            

            plt.savefig(f"{out_base}.pdf", dpi=DPI, format='pdf', bbox_inches='tight', pad_inches=0.1)
            

            svg_text = wc.to_svg(embed_font=True)
            with open(f"{out_base}.svg", "w", encoding="utf-8") as f:
                f.write(svg_text)

            plt.show()

            print("\n" + "="*40)
            print(f"✅ 处理完成！")
            print(f"1. 基因已规范化为小鼠格式 (如 Sox2)")
            print(f"2. 词云已保存为 PNG, PDF 和 纯矢量 SVG 格式")
            print(f"3. 字体成功调用: {FONT_PATH}")
            print("="*40)
            
        except OSError as e:
            print("\n❌ 严重错误: 依然无法加载 Arial 字体。")
            print("请尝试以下终极解决方案：")
            print("Set SAGE_FIGURE_FONT to a TrueType font file if needed.")
            print("Set SAGE_FIGURE_FONT to a TrueType font file if needed.")
            print("3. 再次运行本代码")

1-mouse-gene-ratio-end

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import re
import mygene

# ==========================================

# ==========================================
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']

mpl.rcParams['axes.linewidth'] = 0.5 


def match_keywords(text, keywords):
    for k in keywords:

        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

# ==========================================

# ==========================================
def get_objective_classifications(gene_list):
    if not gene_list:
        print("警告：未读取到任何基因，请检查 txt 文件路径。")
        return {}

    print(f"正在深度联网查询 {len(gene_list)} 个小鼠基因的名称与 GO 功能库，请稍候...")
    mg = mygene.MyGeneInfo()
    

    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='mouse')

    gene_info_dict = {}
    for res in results:
        query_gene = res.get('query')
        if query_gene and query_gene not in gene_info_dict:
            gene_info_dict[query_gene] = res

    classification_dict = {}
    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()


            go_terms = ""
            if 'go' in row:
                go_data = row['go']
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in go_data:
                        items = go_data[sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items:
                                go_terms += str(item.get('term', '')).lower() + " "
            
            search_text = desc + " " + go_terms + " " + gene_type


            

            if 'rna' in gene_type or 'pseudo' in gene_type or re.match(r'^GM\d+$', symbol) or symbol.endswith('RIK') or re.match(r'^BC\d+$', symbol):
                category = 'ncRNA & Uncharacterized'
            

            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or \
                 match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
                

            elif match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
                

            elif match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
                

            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
                

            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
                

            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
                

            elif match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
                

            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
                

            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
                

            elif match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
                

            elif match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
                

            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================

# ==========================================

INPUT_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes") 
OUTPUT_DIR = output_path("2-8.3-shanda/1-feature/1-figure/3-ratio-mouse-gene-tissue")

os.makedirs(OUTPUT_DIR, exist_ok=True)

all_genes = set()
tissue_genes = {}

for file in os.listdir(INPUT_DIR):
    if file.endswith('.txt') and "Knee_Genes" in file:
        tissue_name = re.sub(r'_Knee_Genes_\d+\.txt', '', file).replace('_', ' ')
        with open(os.path.join(INPUT_DIR, file), 'r', encoding='utf-8') as f:
            genes = [line.strip().split()[-1].upper() for line in f.readlines() if line.strip()]
            tissue_genes[tissue_name] = genes
            all_genes.update(genes)

# ==========================================

# ==========================================
global_gene_dict = get_objective_classifications(list(all_genes))

categories = [
    'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
    'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
    'Transcription Regulation', 'Development & Differentiation', 
    'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
    'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
]

results = []
for tissue, genes in tissue_genes.items():
    if not genes: continue
    counts = {cat: 0 for cat in categories}
    for g in genes:
        cat = global_gene_dict.get(g, 'Other / Unclassified')
        counts[cat] += 1

    total = sum(counts.values())
    perc = {cat: (counts[cat]/total)*100 for cat in categories}
    perc['Tissue'] = tissue
    results.append(perc)

df = pd.DataFrame(results)
if df.empty:
    print(f"未生成任何数据，可能是 {INPUT_DIR} 目录下没有找到符合 '_Knee_Genes_' 命名的 txt 文件！")
else:
    df.set_index('Tissue', inplace=True)
    df = df.sort_values(by='Ribosome, RNA & Translation', ascending=False)
    
    df.to_csv(os.path.join(OUTPUT_DIR, "Gene_Classification_Data.csv"))
    pd.DataFrame.from_dict(global_gene_dict, orient='index', columns=['Category']).to_csv(os.path.join(OUTPUT_DIR, "Gene_to_Category_Map.csv"))

    # ==========================================

    # ==========================================

    width_in = 126 / 25.4   
    height_in = 60 / 25.4   
    fig, ax = plt.subplots(figsize=(width_in, height_in))


    colors = [
        '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
        '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
    ]

    ax.yaxis.grid(True, color='#E6E6E6', linestyle='-', linewidth=0.5, zorder=0)


    df[categories].plot(kind='bar', stacked=True, ax=ax, color=colors, 
                        width=0.8, edgecolor='white', linewidth=0.5, zorder=3)

    ax.set_ylim(0, 100)
    

    ax.set_ylabel('Percentage of Genes (%)', fontsize=7, fontweight='normal', labelpad=2)
    ax.set_xlabel('', fontsize=7)
    
    plt.xticks(rotation=45, ha='right', fontsize=6)
    plt.yticks(fontsize=6)
    


    ax.tick_params(axis='y', which='major', width=0.5, length=2.5, pad=2)
    ax.tick_params(axis='x', which='major', width=0.5, length=2.5, pad=0)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


    legend = ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=5.5, 
                       frameon=False, title='Gene Functional Class', title_fontsize=6.5, 
                       labelspacing=0.6, handlelength=1.0, handleheight=0.6)
    legend.get_title().set_fontweight('bold')

    plt.tight_layout()
    
    output_pdf = os.path.join(OUTPUT_DIR, "Aging_Genes_Proportions_Final.pdf")
    output_png = os.path.join(OUTPUT_DIR, "Aging_Genes_Proportions_Final.png")

    output_svg = os.path.join(OUTPUT_DIR, "Aging_Genes_Proportions_Final.svg")
    
    plt.savefig(output_pdf, format='pdf', bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.savefig(output_png, dpi=300, format='png', bbox_inches='tight', pad_inches=0.03, facecolor='white')

    plt.savefig(output_svg, format='svg', bbox_inches='tight', pad_inches=0.03, facecolor='white')

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import Counter
import platform
import matplotlib.font_manager as font_manager
import re
import mygene

# ==========================================

# ==========================================
INPUT_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")
OUTPUT_DIR = output_path("2-8.3-shanda/1-feature/1-figure/2-ciyuntu")

os.makedirs(OUTPUT_DIR, exist_ok=True)


system = platform.system()
if system == "Windows":
    font_path = sage_font_path()
elif system == "Darwin": 
    font_path = sage_font_path()
else: 
    font_path = sage_font_path()
    if not os.path.exists(font_path):
        font_path = sage_font_path()

if os.path.exists(font_path):
    font_manager.fontManager.addfont(font_path)
    mpl.rcParams['font.family'] = 'Arial'
else:
    mpl.rcParams['font.family'] = 'sans-serif'
    mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['axes.linewidth'] = 0.5 

# ==========================================

# ==========================================
def match_keywords(text, keywords):
    for k in keywords:
        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

def get_objective_classifications(gene_list):
    if not gene_list: return {}
    print(f"正在深度联网查询 {len(gene_list)} 个核心基因的 GO 功能库...")
    mg = mygene.MyGeneInfo()
    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='mouse')

    gene_info_dict = {res.get('query'): res for res in results if res.get('query')}
    classification_dict = {}

    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()

            go_terms = ""
            if 'go' in row:
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in row['go']:
                        items = row['go'][sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items: go_terms += str(item.get('term', '')).lower() + " "
            
            search_text = desc + " " + go_terms + " " + gene_type


            if 'rna' in gene_type or 'pseudo' in gene_type or re.match(r'^GM\d+$', symbol) or symbol.endswith('RIK') or re.match(r'^BC\d+$', symbol):
                category = 'ncRNA & Uncharacterized'
            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
            elif match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
            elif match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
            elif match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
            elif match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
            elif match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================

# ==========================================
categories = [
    'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
    'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
    'Transcription Regulation', 'Development & Differentiation', 
    'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
    'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
]
colors = [
    '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
    '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
]
color_map_dict = dict(zip(categories, colors))

# ==========================================

# ==========================================
all_genes = []
for file in os.listdir(INPUT_DIR):
    if file.endswith('.txt'):
        with open(os.path.join(INPUT_DIR, file), 'r', encoding='utf-8') as f:
            for line in f:
                gene = line.strip().upper()

                if gene and "SOURCE" not in gene:
                    all_genes.append(gene)

if not all_genes:
    print(f"❌ 错误: 在 {INPUT_DIR} 中没有提取到任何基因数据。")
    exit()


gene_counter = Counter(all_genes)
top_30 = gene_counter.most_common(30)
top_genes = [item[0] for item in top_30]


classifications = get_objective_classifications(top_genes)


df = pd.DataFrame(top_30, columns=['Gene', 'Frequency'])
df['Category'] = df['Gene'].map(lambda x: classifications.get(x, 'Other / Unclassified'))
df['Color'] = df['Category'].map(color_map_dict)


df['Gene'] = df['Gene'].str.capitalize()


df = df.sort_values(by='Frequency', ascending=True).reset_index(drop=True)


df.to_csv(os.path.join(OUTPUT_DIR, "Top30_Genes_Frequency_Data.csv"), index=False)

# ==========================================

# ==========================================

width_in = 126 / 25.4
height_in = 90 / 25.4
fig, ax = plt.subplots(figsize=(width_in, height_in))


ax.xaxis.grid(True, color='#E6E6E6', linestyle='-', linewidth=0.5, zorder=0)


ax.hlines(y=df.index, xmin=0, xmax=df['Frequency'], color=df['Color'], linewidth=1.0, zorder=3)


ax.scatter(df['Frequency'], df.index, color=df['Color'], s=25, edgecolor='white', linewidth=0.5, zorder=4)


ax.set_yticks(df.index)

ax.set_yticklabels(df['Gene'], fontsize=6, fontstyle='italic')

ax.set_xlabel('Gene Occurrence Frequency', fontsize=7, fontweight='bold', labelpad=4)
ax.tick_params(axis='both', which='major', width=0.5, length=2.5, labelsize=6)


ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)



present_categories = df['Category'].unique()
legend_cats = [c for c in categories if c in present_categories]


legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map_dict[c], markersize=4.5) 
    for c in legend_cats
]

legend = ax.legend(legend_elements, legend_cats, loc='center left', bbox_to_anchor=(1.02, 0.5), 
                   fontsize=5.5, frameon=False, title='Gene Functional Class', title_fontsize=6.5, 
                   labelspacing=0.8, handletextpad=0.2)
legend.get_title().set_fontweight('bold')

plt.tight_layout()


output_pdf = os.path.join(OUTPUT_DIR, "Fig_Top30_Genes_Lollipop.pdf")
output_png = os.path.join(OUTPUT_DIR, "Fig_Top30_Genes_Lollipop.png")

plt.savefig(output_pdf, format='pdf', bbox_inches='tight', pad_inches=0.03)
plt.savefig(output_png, dpi=300, format='png', bbox_inches='tight', pad_inches=0.03)

print("\n" + "="*50)
print(f"✅ 主刊级棒棒糖图 (Lollipop Plot) 生成完毕！")
print(f"📏 物理尺寸: 126 mm x 90 mm")
print(f"📊 已提取 Top 30 基因，并完成斜体 (Italic) 规范化")
print(f"🎨 色板已完美匹配 13 种 NPG 经典功能分类")
print(f"📂 图表与数据 (CSV) 保存至: \n   {OUTPUT_DIR}")
print("="*50)
plt.show()